# Point Cloud to CAD-sequence

In this notebook the complete interactive pipeline for encoding point clouds into a latent space, from which DeepCAD decodes a CAD-sequence.

In [7]:
import os
import sys
import importlib

import torch

sys.path.append("..")
from models.DeepCAD.config.configAE import ConfigAE
from models.DeepCAD.trainer.trainerAE import TrainerAE

### Variables

Store the models in ```experiments```, a results directory will be created for each respective model.

In [2]:
model_name = "best"

### Constants

In [3]:
model_path = os.path.join("experiments", model_name) + ".pth"
results_dir = os.path.join("experiments", model_name + "_results")
if not os.path.exists(results_dir):
    os.mkdir(results_dir)
latent_dim = 256

### Create and load pre-trained PointNet++

In [4]:
def inplace_relu(m):
    classname = m.__class__.__name__
    if classname.find('ReLU') != -1:
        m.inplace=True

sys.path.append(os.path.join('..', 'models','Pointnet_Pointnet2_pytorch', 'models'))
model = importlib.import_module('pointnet2_cls_ssg')
classifier = model.get_model(latent_dim, normal_channel=False)
criterion = model.get_loss_mse()
classifier.apply(inplace_relu)

saved_model = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
state_dict = saved_model['model_state_dict']
if 'module.' in next(iter(state_dict)):
    monitor.log_and_print("Model was saved wrapped in nn.DataParallel.\nRemoving 'module.' from state dict.")
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}#
classifier.load_state_dict(state_dict)
classifier.eval()

<All keys matched successfully>

### Create and load pre-trained DeepCAD

In [9]:
cfg = ConfigAE('test', "../data/latent")
tr_agent = TrainerAE(cfg)
tr_agent.load_ckpt(cfg.ckpt)

----Experiment Configuration-----
proj_dir             /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/notebooks/data/latent
data_root            data
exp_name             pretrained
gpu_ids              0
batch_size           512
num_workers          8
nr_epochs            1000
lr                   0.001
grad_clip            1.0
warmup_step          2000
cont                 False
ckpt                 1000
vis                  False
save_frequency       500
val_frequency        10
vis_frequency        2000
augment              False
mode                 None
outputs              None
z_path               None


ValueError: Checkpoint /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/notebooks/data/latent/pretrained/model/ckpt_epoch1000.pth not exists.

### Gedanken

- ich sollte hier ein Ordner haben in den ich das zu benutzende model setze
- dort werden auch die ergebnisse/visualisierungen abgespeichert